# Investigate the metadata of a protein detective session. 


This notebook uses a session generated by running the commands (first incantation of each command) in the README.md.

In [3]:
from pathlib import Path

session_dir = Path("../mysession")
duckdb_file = (session_dir / "meta.duckdb").absolute()

In [ ]:
from protein_detective.meta import create_meta_duckdb_file

create_meta_duckdb_file(session_dir, duckdb_file=duckdb_file)

## Query duckdb database

Getting a lay of the land by looking at the tables and their relationships in an ER diagram:

```mermaid
erDiagram
  
  "alphafold" {
      VARCHAR uniprot_accession FK
      VARCHAR af_id 
    }
    "uniprot" ||--o{ "alphafold" : has
    "combined_stats" {
      VARCHAR input_file FK
      VARCHAR structure_id 
      VARCHAR uniprot_accession 
      DOUBLE resolution 
      INTEGER high_confidence_residues_count 
      INTEGER total_residue_count 
      VARCHAR method 
      BOOLEAN is_alphafold 
      INTEGER uniprot_start 
      INTEGER uniprot_end 
      DOUBLE sequence_identity 
      INTEGER chain_length 
      DOUBLE geometry_quality 
      BOOLEAN passed 
      VARCHAR output_file FK
      VARCHAR reason 
    }
    "structure_files" ||--o{ "combined_stats" : has
    "structure_files" ||--o{ "combined_stats" : has
    "fittable_structures" {
      VARCHAR structure_file FK
      VARCHAR structure 
      VARCHAR structure_id 
      BOOLEAN is_alphafold 
      VARCHAR uniprot_accessions 
    }
    "structure_files" ||--o{ "fittable_structures" : has
    "pdbe" {
      VARCHAR uniprot_accession FK
      VARCHAR pdb_id 
      VARCHAR method 
      DOUBLE resolution 
      VARCHAR uniprot_chains 
      VARCHAR chain 
      INTEGER chain_length 
    }
    "uniprot" ||--o{ "pdbe" : has
    "rocrate_create_actions" {
      VARCHAR id PK,FK
      VARCHAR type 
      VARCHAR name 
      STRUCT agent 
      VARCHAR endTime 
      STRUCT instrument 
      STRUCT object 
      STRUCT result 
      VARCHAR startTime 
      VARCHAR command 
    }
    "rocrate_nodes" ||--o{ "rocrate_create_actions" : has
    "rocrate_inodes" {
      VARCHAR id PK,FK
      VARCHAR type 
      VARCHAR description 
      VARCHAR name 
    }
    "rocrate_nodes" ||--o{ "rocrate_inodes" : has
    "rocrate_nodes" {
      VARCHAR id PK
      VARCHAR type 
      STRUCT conformsTo 
      VARCHAR datePublished 
      VARCHAR description 
      STRUCT hasPart 
      VARCHAR license 
      VARCHAR name 
      STRUCT about 
      BIGINT contentSize 
      VARCHAR encodingFormat 
      VARCHAR version 
      STRUCT agent 
      VARCHAR endTime 
      STRUCT instrument 
      STRUCT result 
      VARCHAR startTime 
      STRUCT object 
    }

    "rocrate_objects" {
      VARCHAR action_id FK
      VARCHAR object_id FK
    }
    "rocrate_create_actions" ||--o{ "rocrate_objects" : has
    "rocrate_inodes" ||--o{ "rocrate_objects" : has
    "rocrate_results" {
      VARCHAR action_id FK
      VARCHAR result_id FK
    }
    "rocrate_create_actions" ||--o{ "rocrate_results" : has
    "rocrate_inodes" ||--o{ "rocrate_results" : has
    "solutions" {
      VARCHAR powerfit_run_id 
      VARCHAR structure 
      INTEGER rank 
      FLOAT cc 
      FLOAT fishz 
      FLOAT relz 
      FLOAT[3] translation 
      FLOAT[9] rotation 
      VARCHAR template_file FK
      VARCHAR uniprot_accessions 
      VARCHAR structure_id 
      BOOLEAN is_alphafold 
    }
    "structure_files" ||--o{ "solutions" : has
    "structure_files" {
      VARCHAR file PK
      VARCHAR parent_dir 
      VARCHAR filename 
    }

    "uniprot" {
      VARCHAR uniprot_accession PK
    }

    "uniprots_verified_stats" {
      VARCHAR input_file FK
      VARCHAR output_file FK
      BOOLEAN injected 
      VARCHAR uniprot_chain_mappings 
    }
    "structure_files" ||--o{ "uniprots_verified_stats" : has
    "structure_files" ||--o{ "uniprots_verified_stats" : has
```

(This diagram can be generated with `pnpx duckerd -d mysession/meta.duckdb -m mysession/meta.mmd -o /tmp/notneeded.svg` with "meta." prefix removed from table names.)

There are 2 islands
1. `rocrate*` tables extracted from `ro-crate-metadata.json` file and 
2. the rest extracted from filesystem and CSV files.

In [4]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [5]:
import duckdb
import pandas as pd

pd.set_option("display.max_colwidth", None)

%load_ext sql
conn = duckdb.connect(duckdb_file, read_only=True)
%sql conn --alias duckdb

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [6]:
%sql duckdb

Each protein-detective command execution is recorded as an RO-crate create action, which can be queried from the `rocrate_create_actions` table.

In [7]:
%sql SELECT command,startTime,endTime,agent,object,result FROM rocrate_create_actions ORDER BY startTime;

command  \
0  protein-detective search --taxon-id 9606 --reviewed --subcellular-location-uniprot nucleus --subcellular-location-go GO:0005634 --molecular-function-go GO:0003677 --limit-uniprot 100 --pdbe.limit 100 ./mysession   
1                                                                                                                                                                               protein-detective retrieve ./mysession   
2                                                                                                                      protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession   
3                                                                         protein-detective powerfit run --workers-per-gpu 2 --angle 40 --powerfit-run-id myrun1 ../powerfit-tutorial/ribosome-KsgA.map 13 ./mysession   
4                                                                                                                                                                      protein-detective powerfit fit-models mysession   

                          startTime                           endTime  \
0  2026-08-17T09:08:33.997288+00:00  2026-08-17T09:08:42.116271+00:00   
1  2026-08-17T09:08:44.532909+00:00  2026-08-17T09:08:46.842985+00:00   
2  2026-08-17T09:08:51.801345+00:00  2026-08-17T09:08:57.071472+00:00   
3  2026-08-17T09:09:03.781235+00:00  2026-08-17T09:09:30.840095+00:00   
4  2026-08-17T09:09:32.019822+00:00  2026-08-17T09:09:34.074766+00:00   

                agent  \
0  {'@id': 'verhoes'}   
1  {'@id': 'verhoes'}   
2  {'@id': 'verhoes'}   
3  {'@id': 'verhoes'}   
4  {'@id': 'verhoes'}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

Notice that the file paths are relative to the session directory.

## Filter input and output structure files

Lets look at the input structuree files of the filter command.

In [ ]:
%%sql 
SELECT a.command, o.object_id, n.type, n.description , f.file
FROM rocrate_create_actions a 
JOIN rocrate_objects o ON a.id = o.action_id 
JOIN rocrate_inodes n ON o.object_id = n.id
JOIN structure_files f ON n.name = f.parent_dir
WHERE a.command LIKE '%filter%'

,command,object_id,type,description,file
0,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/alphafold/,Dataset,Directory where the AlphaFold files were downloaded.,downloads/alphafold/AF-A0A087WUV0-F1-model_v6.cif.gz
1,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/alphafold/,Dataset,Directory where the AlphaFold files were downloaded.,downloads/alphafold/AF-A0A0C5B5G6-F1-model_v6.cif.gz
2,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/alphafold/,Dataset,Directory where the AlphaFold files were downloaded.,downloads/alphafold/AF-A0A0U1RQI7-F1-model_v6.cif.gz
3,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/alphafold/,Dataset,Directory where the AlphaFold files were downloaded.,downloads/alphafold/AF-A0A1B0GTS1-F1-model_v6.cif.gz
4,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/alphafold/,Dataset,Directory where the AlphaFold files were downloaded.,downloads/alphafold/AF-A0A1B0GVZ6-F1-model_v6.cif.gz
...,...,...,...,...,...
139,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/pdbe/,Dataset,Directory where the PDBe files were downloaded.,downloads/pdbe/6mzd_updated.cif.gz
140,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/pdbe/,Dataset,Directory where the PDBe files were downloaded.,downloads/pdbe/6mzm_updated.cif.gz
141,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/pdbe/,Dataset,Directory where the PDBe files were downloaded.,downloads/pdbe/6nce_updated.cif.gz
142,protein-detective filter --min-confidence 50 --min-residues 100 --max-residues 1000 ./mysession,downloads/pdbe/,Dataset,Directory where the PDBe files were downloaded.,downloads/pdbe/6ncm_updated.cif.gz


Similar but for output structure files of the filter command. Above we can see that the filter command outputted a 'combined_stats.csv' file, which can be queried using the `combined_stats` table.

Lets first look at why 1H3O did not pass the filter.

In [31]:
%%sql 
SELECT * FROM combined_stats WHERE structure_id ='1H3O';


,input_file,structure_id,uniprot_accession,resolution,high_confidence_residues_count,total_residue_count,method,is_alphafold,uniprot_start,uniprot_end,sequence_identity,chain_length,geometry_quality,passed,output_file,reason
0,combined_input/1h3o_updated_A2A.cif.gz,1H3O,O00268,2.3,<NA>,50,X-ray,False,870,943,1.0,50,8.53,False,None,"Chain length 50 not in range [100, 1000]"


Ah, it has a residue count of 50 while filter had 100..1000 as valid range.

Lets look at a passing structure

In [35]:
%sql SELECT * FROM combined_stats WHERE passed=True LIMIT 1;

,input_file,structure_id,uniprot_accession,resolution,high_confidence_residues_count,total_residue_count,method,is_alphafold,uniprot_start,uniprot_end,sequence_identity,chain_length,geometry_quality,passed,output_file,reason
0,combined_input/AF-A0A087WUV0-F1-model_v6.cif.gz,AF-A0A087WUV0-F1,A0A087WUV0,0.0,328,522,Predicted,True,1,522,1.0,522,NaN,True,combined_output/AF-A0A087WUV0-F1-model_v6.cif.gz,None


## Powerfit solutions

Lets get the top 10 best overall ranked solutions taking the best ranked solution for each structure.

In [41]:
%%sql
SELECT * FROM solutions WHERE rank=1 ORDER BY cc DESC LIMIT 10;


,powerfit_run_id,structure,rank,cc,fishz,relz,translation,rotation,template_file,uniprot_accessions,structure_id,is_alphafold
0,myrun1,3i8z_updated_A2A.cif.gz,1,0.598,0.690,12.959000,"[239.46, 187.27, 211.83]","[-0.238, 0.322, 0.916, 0.916, -0.238, 0.322, 0.322, 0.916, -0.238]",combined_output/3i8z_updated_A2A.cif.gz,O00257,3I8Z,False
1,myrun1,6mzc_updated_E2A.cif.gz,1,0.547,0.614,14.671000,"[199.55, 214.9, 165.78]","[1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, -1.0]",combined_output/6mzc_updated_E2A.cif.gz,O00268,6MZC,False
2,myrun1,6mzd_updated_D2A.cif.gz,1,0.514,0.568,13.612000,"[273.23, 251.74, 165.78]","[0.916, 0.238, -0.322, -0.238, -0.322, -0.916, -0.322, 0.916, -0.238]",combined_output/6mzd_updated_D2A.cif.gz,O00268,6MZD,False
3,myrun1,AF-O00110-F1-model_v6.cif.gz,1,0.471,0.512,14.765000,"[236.39, 236.39, 227.18]","[0.0, 1.0, 0.0, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0]",combined_output/AF-O00110-F1-model_v6.cif.gz,O00110,AF-O00110-F1,True
4,myrun1,AF-A0A2R8Y619-F1-model_v6.cif.gz,1,0.463,0.501,13.182000,"[221.04, 227.18, 190.34]","[0.0, -1.0, 0.0, 0.0, 0.0, 1.0, -1.0, 0.0, 0.0]",combined_output/AF-A0A2R8Y619-F1-model_v6.cif.gz,A0A2R8Y619,AF-A0A2R8Y619-F1,True
5,myrun1,AF-A0A1W2PPK0-F1-model_v6.cif.gz,1,0.459,0.496,13.738000,"[174.99, 221.04, 171.92]","[-0.0, 0.0, 1.0, 0.0, -1.0, 0.0, 1.0, 0.0, -0.0]",combined_output/AF-A0A1W2PPK0-F1-model_v6.cif.gz,A0A1W2PPK0,AF-A0A1W2PPK0-F1,True
6,myrun1,AF-A6NDR6-F1-model_v6.cif.gz,1,0.458,0.495,17.065001,"[132.01, 150.43, 211.83]","[0.322, -0.916, 0.238, 0.238, 0.322, 0.916, -0.916, -0.238, 0.322]",combined_output/AF-A6NDR6-F1-model_v6.cif.gz,A6NDR6,AF-A6NDR6-F1,True
7,myrun1,AF-A0A5F9ZHS7-F1-model_v6.cif.gz,1,0.452,0.487,15.162000,"[285.51, 248.67, 156.57]","[0.0, -1.0, 0.0, 0.0, 0.0, -1.0, 1.0, 0.0, 0.0]",combined_output/AF-A0A5F9ZHS7-F1-model_v6.cif.gz,A0A5F9ZHS7,AF-A0A5F9ZHS7-F1,True
8,myrun1,AF-E9PAV3-F1-model_v6.cif.gz,1,0.450,0.485,14.297000,"[254.81, 224.11, 181.13]","[-0.322, 0.916, 0.238, -0.916, -0.238, -0.322, -0.238, -0.322, 0.916]",combined_output/AF-E9PAV3-F1-model_v6.cif.gz,E9PAV3,AF-E9PAV3-F1,True
9,myrun1,6mew_updated_A2A.cif.gz,1,0.447,0.481,15.707000,"[187.27, 205.69, 254.81]","[0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]",combined_output/6mew_updated_A2A.cif.gz,O14593,6MEW,False


Lets say that I know the density map actually is a protein with uniprot accession `O14593`, and I want to check that it is in the top 10 best overall ranked solutions. 

In [ ]:
# TODO